In [1]:
import argparse
import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm


DISEASE_COLS = [
    "No Finding",
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
]

PATHOLOGY_COLS = [col for col in DISEASE_COLS if col != "No Finding"]


def parse_args():
    parser = argparse.ArgumentParser(
        description="Evaluate all saved U-Ones checkpoints with classification and calibration metrics."
    )
    parser.add_argument(
        "--csv-path",
        default="train_cheXbert.csv",
        help="Path to the training CSV used in the notebook.",
    )
    parser.add_argument(
        "--checkpoints-dir",
        default=".",
        help="Directory containing the four saved model checkpoints.",
    )
    parser.add_argument(
        "--output-dir",
        default="evaluation_outputs_uones",
        help="Directory where evaluation tables and plots will be saved.",
    )
    parser.add_argument(
        "--base-dir",
        default="/home/ubuntu/chexpert/chexpertchestxrays-u20210408",
        help="Base directory that contains the CheXpert batch folders.",
    )
    parser.add_argument(
        "--num-workers",
        type=int,
        default=4,
        help="Number of DataLoader workers for evaluation.",
    )
    parser.add_argument(
        "--batch-size-single",
        type=int,
        default=128,
        help="Batch size for single-view model evaluation.",
    )
    parser.add_argument(
        "--batch-size-dual",
        type=int,
        default=8,
        help="Batch size for dual-view model evaluation.",
    )
    args, _ = parser.parse_known_args()
    return args


def build_batch_dirs(base_dir):
    return [
        f"{base_dir}/CheXpert-v1.0 batch 2 (train 1)",
        f"{base_dir}/CheXpert-v1.0 batch 3 (train 2)",
        f"{base_dir}/CheXpert-v1.0 batch 4 (train 3)",
    ]


def resolve_path(csv_path, batch_dirs):
    relative = csv_path.replace("CheXpert-v1.0/train/", "")
    for batch_dir in batch_dirs:
        candidate = os.path.join(batch_dir, relative)
        if os.path.exists(candidate):
            return candidate
    return None


def prepare_uones_dataframe(csv_path):
    train_df = pd.read_csv(csv_path)
    train_df["Patient_ID"] = train_df["Path"].str.extract(r"(patient\d+)")

    if "Support Devices" in train_df.columns:
        train_df = train_df.drop(columns=["Support Devices"])

    images_per_patient = train_df["Patient_ID"].value_counts()
    patients_more_than_30 = images_per_patient[images_per_patient > 30].index
    train_df = train_df[~train_df["Patient_ID"].isin(patients_more_than_30)].copy()

    base_df = train_df.copy()
    base_df[DISEASE_COLS] = base_df[DISEASE_COLS].fillna(0)

    u_ones_df = base_df.copy()
    u_ones_df[PATHOLOGY_COLS] = u_ones_df[PATHOLOGY_COLS].replace(-1, 1)
    return u_ones_df


def build_single_view_split(u_ones_df):
    model_df = u_ones_df[u_ones_df["Frontal/Lateral"] == "Frontal"].copy()
    patient_ids = model_df["Patient_ID"].dropna().unique()

    rng = np.random.RandomState(42)
    shuffled = patient_ids.copy()
    rng.shuffle(shuffled)

    split_idx = int(len(shuffled) * 0.8)
    train_patients = shuffled[:split_idx]
    val_patients = shuffled[split_idx:]

    train_split_df = model_df[model_df["Patient_ID"].isin(train_patients)].copy()
    val_split_df = model_df[model_df["Patient_ID"].isin(val_patients)].copy()
    return train_split_df, val_split_df


def build_dual_view_split(u_ones_df):
    model_df = u_ones_df.copy()
    model_df["Study_ID"] = model_df["Path"].str.extract(r"(patient\d+/study\d+)")

    frontal_df = model_df[model_df["Frontal/Lateral"] == "Frontal"].copy()
    lateral_df = model_df[model_df["Frontal/Lateral"] == "Lateral"].copy()

    study_df = frontal_df.merge(
        lateral_df[["Study_ID", "Path"]],
        on="Study_ID",
        how="inner",
        suffixes=("_frontal", "_lateral"),
    )
    study_df["Patient_ID"] = study_df["Study_ID"].str.extract(r"(patient\d+)")

    patient_ids = study_df["Patient_ID"].dropna().unique()
    rng = np.random.RandomState(42)
    shuffled = patient_ids.copy()
    rng.shuffle(shuffled)

    split_idx = int(len(shuffled) * 0.8)
    train_patients = shuffled[:split_idx]
    val_patients = shuffled[split_idx:]

    train_split_df = study_df[study_df["Patient_ID"].isin(train_patients)].copy()
    val_split_df = study_df[study_df["Patient_ID"].isin(val_patients)].copy()
    return train_split_df, val_split_df


def get_single_view_transform():
    return transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ]
    )


def get_dual_view_transform():
    return transforms.Compose(
        [
            transforms.Resize((192, 192)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ]
    )


class CheXpertDataset(Dataset):
    def __init__(self, df, label_cols, batch_dirs, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_cols = label_cols
        self.batch_dirs = batch_dirs
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        rel_path = self.df.loc[idx, "Path"]
        img_path = resolve_path(rel_path, self.batch_dirs)

        try:
            image = Image.open(img_path).convert("RGB")
        except (FileNotFoundError, OSError, TypeError, AttributeError):
            image = Image.new("RGB", (224, 224), 0)

        if self.transform:
            image = self.transform(image)

        labels = self.df.loc[idx, self.label_cols].values.astype(np.float32)
        return image, torch.tensor(labels, dtype=torch.float32)


class DualViewCheXpertDataset(Dataset):
    def __init__(self, df, label_cols, batch_dirs, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_cols = label_cols
        self.batch_dirs = batch_dirs
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        frontal_rel = self.df.loc[idx, "Path_frontal"]
        lateral_rel = self.df.loc[idx, "Path_lateral"]

        frontal_path = resolve_path(frontal_rel, self.batch_dirs)
        lateral_path = resolve_path(lateral_rel, self.batch_dirs)

        try:
            frontal_img = Image.open(frontal_path).convert("RGB")
        except (FileNotFoundError, OSError, TypeError, AttributeError):
            frontal_img = Image.new("RGB", (192, 192), 0)

        try:
            lateral_img = Image.open(lateral_path).convert("RGB")
        except (FileNotFoundError, OSError, TypeError, AttributeError):
            lateral_img = Image.new("RGB", (192, 192), 0)

        if self.transform:
            frontal_img = self.transform(frontal_img)
            lateral_img = self.transform(lateral_img)

        labels = self.df.loc[idx, self.label_cols].values.astype(np.float32)
        labels = torch.tensor(labels, dtype=torch.float32)
        return frontal_img, lateral_img, labels


class CaMCheXDualView(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        frontal_backbone = models.convnext_tiny(
            weights=models.ConvNeXt_Tiny_Weights.DEFAULT
        )
        lateral_backbone = models.convnext_tiny(
            weights=models.ConvNeXt_Tiny_Weights.DEFAULT
        )

        frontal_in_features = frontal_backbone.classifier[2].in_features
        lateral_in_features = lateral_backbone.classifier[2].in_features

        frontal_backbone.classifier = nn.Identity()
        lateral_backbone.classifier = nn.Identity()

        self.frontal_encoder = frontal_backbone
        self.lateral_encoder = lateral_backbone

        fusion_dim = frontal_in_features + lateral_in_features

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, frontal_img, lateral_img):
        frontal_feat = self.frontal_encoder(frontal_img)
        lateral_feat = self.lateral_encoder(lateral_img)

        frontal_feat = torch.flatten(frontal_feat, 1)
        lateral_feat = torch.flatten(lateral_feat, 1)

        fused = torch.cat([frontal_feat, lateral_feat], dim=1)
        return self.classifier(fused)


class MultiViewDualBranchNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        frontal_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        frontal_in_features = frontal_backbone.fc.in_features
        frontal_backbone.fc = nn.Identity()

        lateral_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        lateral_in_features = lateral_backbone.fc.in_features
        lateral_backbone.fc = nn.Identity()

        self.frontal_branch = frontal_backbone
        self.lateral_branch = lateral_backbone

        fusion_dim = frontal_in_features + lateral_in_features

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, frontal_img, lateral_img):
        frontal_feat = self.frontal_branch(frontal_img)
        lateral_feat = self.lateral_branch(lateral_img)
        fused = torch.cat((frontal_feat, lateral_feat), dim=1)
        return self.classifier(fused)


def build_resnet50(num_classes):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


def build_densenet121(num_classes):
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    return model


def evaluate_single_view(model, loader, device):
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            probs = torch.sigmoid(logits)

            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())

    labels = torch.cat(all_labels, dim=0).numpy()
    probs = torch.cat(all_probs, dim=0).numpy()
    return labels, probs


def evaluate_dual_view(model, loader, device):
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for frontal_imgs, lateral_imgs, labels in tqdm(loader, desc="Evaluating", leave=False):
            frontal_imgs = frontal_imgs.to(device, non_blocking=True)
            lateral_imgs = lateral_imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(frontal_imgs, lateral_imgs)
            probs = torch.sigmoid(logits)

            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())

    labels = torch.cat(all_labels, dim=0).numpy()
    probs = torch.cat(all_probs, dim=0).numpy()
    return labels, probs


def compute_calibration_bins(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    bin_accs = []
    bin_confs = []
    bin_counts = []

    ece = 0.0
    mce = 0.0

    for b in range(n_bins):
        mask = bin_ids == b
        if np.sum(mask) > 0:
            acc = np.mean(y_true[mask])
            conf = np.mean(y_prob[mask])
            frac = np.mean(mask)
            gap = abs(acc - conf)

            ece += gap * frac
            mce = max(mce, gap)
            bin_accs.append(acc)
            bin_confs.append(conf)
            bin_counts.append(np.sum(mask))
        else:
            bin_accs.append(0.0)
            bin_confs.append(0.0)
            bin_counts.append(0)

    return {
        "ece": float(ece),
        "mce": float(mce),
        "bins": bins,
        "bin_accs": np.array(bin_accs),
        "bin_confs": np.array(bin_confs),
        "bin_counts": np.array(bin_counts),
    }


def safe_roc_auc(y_true, y_prob):
    try:
        return float(roc_auc_score(y_true, y_prob))
    except ValueError:
        return math.nan


def safe_auprc(y_true, y_prob):
    try:
        return float(average_precision_score(y_true, y_prob))
    except ValueError:
        return math.nan


def safe_log_loss(y_true, y_prob):
    clipped = np.clip(y_prob, 1e-7, 1 - 1e-7)
    try:
        return float(log_loss(y_true, clipped, labels=[0, 1]))
    except ValueError:
        return math.nan


def compute_metrics(labels, probs):
    preds = (probs >= 0.5).astype(int)
    per_disease_rows = []

    for i, disease in enumerate(DISEASE_COLS):
        y_true = labels[:, i]
        y_prob = probs[:, i]
        y_pred = preds[:, i]

        prevalence = float(np.mean(y_true))
        brier = float(np.mean((y_prob - y_true) ** 2))
        nll = safe_log_loss(y_true, y_prob)

        calibration = compute_calibration_bins(y_true, y_prob, n_bins=10)

        per_disease_rows.append(
            {
                "Disease": disease,
                "Prevalence": prevalence,
                "Accuracy": float(accuracy_score(y_true, y_pred)),
                "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
                "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
                "F1 Score": float(f1_score(y_true, y_pred, zero_division=0)),
                "AUROC": safe_roc_auc(y_true, y_prob),
                "AUPRC": safe_auprc(y_true, y_prob),
                "Brier Score": brier,
                "ECE": calibration["ece"],
                "MCE": calibration["mce"],
                "Log Loss": nll,
            }
        )

    metrics_df = pd.DataFrame(per_disease_rows)

    flat_labels = labels.flatten()
    flat_probs = probs.flatten()
    flat_preds = preds.flatten()
    calibration = compute_calibration_bins(flat_labels, flat_probs, n_bins=10)

    overall = {
        "micro_accuracy": float(accuracy_score(flat_labels, flat_preds)),
        "micro_precision": float(precision_score(flat_labels, flat_preds, zero_division=0)),
        "micro_recall": float(recall_score(flat_labels, flat_preds, zero_division=0)),
        "micro_f1": float(f1_score(flat_labels, flat_preds, zero_division=0)),
        "macro_accuracy": float(metrics_df["Accuracy"].mean()),
        "macro_precision": float(metrics_df["Precision"].mean()),
        "macro_recall": float(metrics_df["Recall"].mean()),
        "macro_f1": float(metrics_df["F1 Score"].mean()),
        "macro_auroc": float(np.nanmean(metrics_df["AUROC"])),
        "micro_auroc": safe_roc_auc(flat_labels, flat_probs),
        "macro_auprc": float(np.nanmean(metrics_df["AUPRC"])),
        "micro_auprc": safe_auprc(flat_labels, flat_probs),
        "micro_brier_score": float(np.mean((flat_probs - flat_labels) ** 2)),
        "micro_ece": calibration["ece"],
        "micro_mce": calibration["mce"],
        "micro_log_loss": safe_log_loss(flat_labels, flat_probs),
    }
    return metrics_df, overall, calibration


def plot_multilabel_roc(labels, probs, title, output_path):
    fig, axes = plt.subplots(4, 4, figsize=(18, 18))
    axes = axes.flatten()

    for i, disease in enumerate(DISEASE_COLS):
        ax = axes[i]
        y_true = labels[:, i]
        y_prob = probs[:, i]

        try:
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            auroc = roc_auc_score(y_true, y_prob)
            ax.plot(fpr, tpr, label=f"AUROC = {auroc:.3f}")
        except ValueError:
            ax.text(0.5, 0.5, "ROC unavailable", ha="center", va="center")

        ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax.set_title(disease)
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend(loc="lower right")

    for j in range(len(DISEASE_COLS), len(axes)):
        axes[j].axis("off")

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_multilabel_pr(labels, probs, title, output_path):
    fig, axes = plt.subplots(4, 4, figsize=(18, 18))
    axes = axes.flatten()

    for i, disease in enumerate(DISEASE_COLS):
        ax = axes[i]
        y_true = labels[:, i]
        y_prob = probs[:, i]

        try:
            precision, recall, _ = precision_recall_curve(y_true, y_prob)
            auprc = average_precision_score(y_true, y_prob)
            ax.plot(recall, precision, label=f"AUPRC = {auprc:.3f}")
        except ValueError:
            ax.text(0.5, 0.5, "PR unavailable", ha="center", va="center")

        baseline = np.mean(y_true)
        ax.axhline(baseline, linestyle="--", color="gray", label=f"Base = {baseline:.3f}")
        ax.set_title(disease)
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.legend(loc="lower left")

    for j in range(len(DISEASE_COLS), len(axes)):
        axes[j].axis("off")

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_reliability_diagram(calibration, title, output_path):
    bins = calibration["bins"]
    bin_accs = calibration["bin_accs"]
    bin_centers = (bins[:-1] + bins[1:]) / 2

    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], linestyle="--", color="black")
    plt.bar(bin_centers, bin_accs, width=0.08, alpha=0.7, edgecolor="black")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()


def plot_confidence_histogram(calibration, title, output_path):
    bins = calibration["bins"]
    counts = calibration["bin_counts"]
    bin_centers = (bins[:-1] + bins[1:]) / 2

    plt.figure(figsize=(7, 4))
    plt.bar(bin_centers, counts, width=0.08, alpha=0.8, edgecolor="black")
    plt.xlabel("Confidence Bin")
    plt.ylabel("Count")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()


def save_outputs(model_name, labels, probs, metrics_df, overall, calibration, output_dir):
    model_dir = output_dir / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    metrics_df.to_csv(model_dir / "per_disease_metrics.csv", index=False)
    pd.DataFrame([overall]).to_csv(model_dir / "overall_metrics.csv", index=False)

    with open(model_dir / "overall_metrics.json", "w") as f:
        json.dump(overall, f, indent=2)

    np.savez_compressed(
        model_dir / "predictions.npz",
        labels=labels,
        probabilities=probs,
    )

    plot_multilabel_roc(
        labels,
        probs,
        title=f"ROC Curves - {model_name}",
        output_path=model_dir / "roc_curves.png",
    )
    plot_multilabel_pr(
        labels,
        probs,
        title=f"Precision-Recall Curves - {model_name}",
        output_path=model_dir / "pr_curves.png",
    )
    plot_reliability_diagram(
        calibration,
        title=f"Reliability Diagram - {model_name}",
        output_path=model_dir / "reliability_diagram.png",
    )
    plot_confidence_histogram(
        calibration,
        title=f"Confidence Histogram - {model_name}",
        output_path=model_dir / "confidence_histogram.png",
    )


def load_checkpoint(model, checkpoint_path, device):
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    return model


def evaluate_model(model_name, model, checkpoint_path, loader, device, output_dir, dual_view):
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")

    model = load_checkpoint(model, checkpoint_path, device)
    labels, probs = (
        evaluate_dual_view(model, loader, device)
        if dual_view
        else evaluate_single_view(model, loader, device)
    )

    metrics_df, overall, calibration = compute_metrics(labels, probs)
    save_outputs(model_name, labels, probs, metrics_df, overall, calibration, output_dir)

    print(f"\n{model_name}")
    print("-" * len(model_name))
    print(f"Macro AUROC : {overall['macro_auroc']:.4f}")
    print(f"Micro AUROC : {overall['micro_auroc']:.4f}")
    print(f"Macro AUPRC : {overall['macro_auprc']:.4f}")
    print(f"Micro AUPRC : {overall['micro_auprc']:.4f}")
    print(f"Micro ECE   : {overall['micro_ece']:.4f}")
    print(f"Micro MCE   : {overall['micro_mce']:.4f}")
    print(f"Brier Score : {overall['micro_brier_score']:.4f}")


def main():
    args = parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoints_dir = Path(args.checkpoints_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    batch_dirs = build_batch_dirs(args.base_dir)
    u_ones_df = prepare_uones_dataframe(args.csv_path)

    _, single_val_df = build_single_view_split(u_ones_df)
    _, dual_val_df = build_dual_view_split(u_ones_df)

    single_loader = DataLoader(
        CheXpertDataset(
            df=single_val_df,
            label_cols=DISEASE_COLS,
            batch_dirs=batch_dirs,
            transform=get_single_view_transform(),
        ),
        batch_size=args.batch_size_single,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    dual_loader = DataLoader(
        DualViewCheXpertDataset(
            df=dual_val_df,
            label_cols=DISEASE_COLS,
            batch_dirs=batch_dirs,
            transform=get_dual_view_transform(),
        ),
        batch_size=args.batch_size_dual,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    evaluations = [
        (
            "resnet50_uones",
            build_resnet50(len(DISEASE_COLS)),
            checkpoints_dir / "best_resnet50_uones.pth",
            single_loader,
            False,
        ),
        (
            "densenet121_uones",
            build_densenet121(len(DISEASE_COLS)),
            checkpoints_dir / "best_densenet121_uones.pth",
            single_loader,
            False,
        ),
        (
            "camchex_dual_view_uones",
            CaMCheXDualView(len(DISEASE_COLS)),
            checkpoints_dir / "best_camchex_uones.pth",
            dual_loader,
            True,
        ),
        (
            "multiview_dual_branch_uones",
            MultiViewDualBranchNet(len(DISEASE_COLS)),
            checkpoints_dir / "best_multiview_dual_branch_uones.pth",
            dual_loader,
            True,
        ),
    ]

    summary_rows = []

    for model_name, model, checkpoint_path, loader, dual_view in evaluations:
        evaluate_model(
            model_name=model_name,
            model=model,
            checkpoint_path=checkpoint_path,
            loader=loader,
            device=device,
            output_dir=output_dir,
            dual_view=dual_view,
        )

        overall_df = pd.read_csv(output_dir / model_name / "overall_metrics.csv")
        row = overall_df.iloc[0].to_dict()
        row["model"] = model_name
        summary_rows.append(row)

    pd.DataFrame(summary_rows).to_csv(output_dir / "model_summary.csv", index=False)
    print(f"\nSaved full evaluation outputs to: {output_dir.resolve()}")


if __name__ == "__main__":
    main()


/tmp/ipykernel_19070/1420708144.py:136: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(shuffled)
/tmp/ipykernel_19070/1420708144.py:165: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(shuffled)
                                                             


resnet50_uones
--------------
Macro AUROC : 0.7720
Micro AUROC : 0.8498
Macro AUPRC : 0.4115
Micro AUPRC : 0.6065
Micro ECE   : 0.0189
Micro MCE   : 0.0628
Brier Score : 0.1061



densenet121_uones
-----------------
Macro AUROC : 0.7647
Micro AUROC : 0.8468
Macro AUPRC : 0.4025
Micro AUPRC : 0.6013
Micro ECE   : 0.0238
Micro MCE   : 0.0677
Brier Score : 0.1073



camchex_dual_view_uones
-----------------------
Macro AUROC : 0.7396
Micro AUROC : 0.7871
Macro AUPRC : 0.3546
Micro AUPRC : 0.4758
Micro ECE   : 0.0309
Micro MCE   : 0.0565
Brier Score : 0.1033



multiview_dual_branch_uones
---------------------------
Macro AUROC : 0.7507
Micro AUROC : 0.7861
Macro AUPRC : 0.3694
Micro AUPRC : 0.4903
Micro ECE   : 0.0196
Micro MCE   : 0.0256
Brier Score : 0.1013

Saved full evaluation outputs to: /home/ubuntu/chexpert/chexpertchestxrays-u20210408/evaluation_outputs_uones


### Comparision Summary:
- Uones demonstrated competitive AUPRC performance, with ResNet-50 achieving the highest Macro AUPRC of 0.4115 across both strategies, indicating better precision-recall balance for rare disease detection.
- The Multiview Dual Branch Network showed the best MCE of 0.0256 under U-Ones, reflecting well-controlled maximum calibration error. 
- Overall, U-Ones represents a clinically conservative approach that prioritizes sensitivity over specificity, and while it underperforms U-Zeros on aggregate discrimination metrics, its AUPRC advantage on minority classes makes it a relevant comparison point for uncertainty-aware medical imaging systems.